#  DelayPredict — Model Review & Selection

**Milestone 03 | AIOps SoSe 2026 | Hochschule Heilbronn**

---

## 1. All Runs Overview

The following table summarizes all models tracked in the `DelayPredict` MLflow experiment,
ordered by ROC-AUC (primary metric).

| Model | Accuracy | F1 | Precision | Recall | ROC-AUC |
|---|---|---|---|---|---|
| **XGBoost_NativeCategorical** | 0.6711 | 0.5856 | 0.6673 | 0.5218 | **0.7255** |
| XGBoost_v2_FeatureExperiment | 0.6714 | 0.5856 | 0.6682 | 0.5212 | 0.7238 |
| RandomForest_TargetEncoding | 0.6670 | 0.5717 | 0.6693 | 0.4990 | 0.7205 |
| DecisionTree_AirlineRoute3to1 | 0.6549 | 0.5707 | 0.6400 | 0.5150 | 0.7026 |
| DecisionTree_FeatureHashing | 0.6463 | 0.5656 | 0.6243 | 0.5170 | 0.6920 |
| DecisionTree_TargetEncoding | 0.6467 | 0.5678 | 0.6238 | 0.5210 | 0.6890 |
| LogisticRegression *(baseline)* | 0.6464 | 0.5506 | 0.6345 | 0.4862 | 0.6921 |
| DummyClassifier *(floor)* | 0.5546 | 0.0000 | 0.0000 | 0.0000 | 0.5000 |

---

## 2. Metric Choice: Why ROC-AUC?

ROC-AUC is the primary selection metric for this binary classification problem for two reasons:

- **Class imbalance**: the dataset is ~55/45 (no delay / delay), so accuracy alone is misleading — the DummyClassifier already reaches 55.5% accuracy by predicting the majority class.
- **Threshold independence**: ROC-AUC measures the model's ability to rank flights correctly across all possible thresholds, which is more informative than any single threshold decision.

F1 is used as a secondary check to ensure a reasonable balance between precision and recall.

---

## 3. Selected Model: XGBoost (NativeCategorical)

**Selected run:** `XGBoost_NativeCategorical` (notebook `03b_xgboost.ipynb`)

| Metric | Baseline (LR) | Selected (XGBoost) | Delta |
|---|---|---|---|
| Accuracy | 0.6464 | 0.6711 | +0.0247 |
| F1 | 0.5506 | 0.5856 | +0.0350 |
| ROC-AUC | 0.6921 | 0.7255 | +0.0334 |

### Why XGBoost over the alternatives?

**vs. XGBoost_v2_FeatureExperiment:** Nearly identical performance (ROC-AUC 0.7255 vs. 0.7238),
but `NativeCategorical` is simpler — no additional feature engineering, fewer moving parts, easier to maintain and reproduce.

**vs. RandomForest:** XGBoost achieves higher ROC-AUC (0.7255 vs. 0.7205) and better F1 (0.5856 vs. 0.5717) with a significantly shorter training time (4s vs. 21s). Sequential boosting outperforms bagging on this tabular dataset.

**vs. Decision Trees:** All three Decision Tree variants underperform across all metrics. The best DT variant (AirlineRoute3to1, ROC-AUC 0.7026) still trails XGBoost by 0.023 in ROC-AUC. The interpretability advantage of single trees does not justify the performance loss here.

**vs. Logistic Regression (baseline):** XGBoost improves ROC-AUC by +0.033 and F1 by +0.035, confirming that the non-linear ensemble approach captures patterns the linear model cannot.

---

## 4. Acknowledged Trade-offs

| Trade-off | Details |
|---|---|
| **Interpretability** | XGBoost is less interpretable than Logistic Regression or Decision Trees. Mitigated by SHAP / feature importance analysis in the notebook. |
| **Training complexity** | More hyperparameters than LR. For this project, default tuning (n_estimators=500, max_depth=6) proved sufficient. |
| **Marginal gain over RF** | ROC-AUC improvement over Random Forest is +0.005. The gain is real but modest — RF would be a defensible alternative if training time were a constraint. |

---

## 5. Conclusion

`XGBoost_NativeCategorical` is selected as the final model. It achieves the highest ROC-AUC (0.7255)
and F1 (0.5856) across all tracked runs, beats the baseline by a clear margin, and trains in under 5 seconds.
The native categorical encoding keeps the pipeline simple and self-contained.

The trained model artifact is logged in MLflow under the `DelayPredict` experiment and saved
as `models/xgb_model.pkl` for deployment.